# Polaris: Gemma 4 Decision Lab

### An executed, reproducible companion to the live Polaris application

**Team:** Imtiaz Hossain · Mofftasim Hossain Sayem  
**Track:** Open Innovation  
**Live demo:** https://polaris-gemma4.vercel.app/demo  
**Repository:** https://github.com/ImtiazHossain-Eshan/polaris-gemma4

Polaris is not a writing assistant. It turns a student's profile, constraints, evidence, and deadlines into a measurable roadmap—then adapts that roadmap when reality changes. This notebook exposes the model-facing workflow that the web application packages into an interactive experience.

## What this notebook proves

1. **Gemma 4 is the only generative model.** The model identifier is fixed in code.
2. **Retrieval and scoring are deterministic.** They supply inspectable context; they do not replace the model.
3. **Gemma 4 performs the central reasoning.** It prioritizes milestones, explains trade-offs, audits evidence, and produces Bengali guidance.
4. **Outputs are machine-checkable.** Every structured response is parsed and evaluated.
5. **The Decision Twin reacts to changed constraints.** A student can see what moves, why it moves, and what evidence to collect next.

```text
Student profile + changed constraint
                │
                ▼
      deterministic retrieval
                │
                ▼
    compact context + JSON contract
                │
                ▼
             Gemma 4
                │
      ┌─────────┼─────────┐
      ▼         ▼         ▼
   roadmap   plan diff   evidence graph
      │         │         │
      └─────────┴─────────┘
                ▼
      validated Action Lab UI
```

## 1. Environment

On Kaggle, store the key as a private secret named `GEMMA_API_KEY`. The cell also accepts the same environment variable locally. The secret is never printed or written into this notebook.

In [1]:
# Kaggle already provides Python. Uncomment only if the SDK is unavailable.
# !pip install -q google-genai

import json
import os
import re
from collections import Counter
from math import log

from google import genai
from google.genai import types

def read_secret(name: str) -> str:
    value = os.environ.get(name, "")
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""

API_KEY = read_secret("GEMMA_API_KEY")
MODEL = os.environ.get("GEMMA_MODEL", "gemma-4-26b-a4b-it")
assert API_KEY, "Add GEMMA_API_KEY through Kaggle Secrets before running."
assert MODEL in {"gemma-4-26b-a4b-it", "gemma-4-31b-it"}

client = genai.Client(api_key=API_KEY)
print("Gemma client ready")
print("Model:", MODEL)
print("Credential present:", bool(API_KEY), "(value hidden)")

Gemma client ready
Model: gemma-4-26b-a4b-it
Credential present: True (value hidden)


## 2. Bangladesh-context student and evidence base

The sample is intentionally concrete: a Bangladeshi HSC student targeting competitive Computer Science programs with a real score gap, limited weekly time, and a funding constraint.

In [2]:
student = {
    "country": "Bangladesh",
    "stage": "HSC / Class 12",
    "target_degree": "Computer Science undergraduate",
    "target_countries": ["United States", "Canada"],
    "gpa": 3.80,
    "sat": 1320,
    "sat_target": 1500,
    "ielts": 6.5,
    "weekly_hours": 14,
    "budget_bdt": 180_000,
    "strengths": ["one deployed student portal", "school club leadership"],
    "gaps": ["testing", "research evidence", "measured project impact"],
}

knowledge_base = [
    {
        "id": "testing",
        "title": "Testing evidence",
        "text": "Use timed diagnostics, keep an error log by skill, and retest after a focused practice cycle. A score without dated practice evidence is not an actionable plan.",
    },
    {
        "id": "projects",
        "title": "Project evidence",
        "text": "Admissions readers can verify shipped work through public artifacts, repository history, user adoption, independent references, and measured outcomes.",
    },
    {
        "id": "funding",
        "title": "Funding feasibility",
        "text": "For international applicants, funding research must begin with the university list. Track eligibility, total cost after aid, required essays, and scholarship deadlines.",
    },
    {
        "id": "bangladesh",
        "title": "Bangladesh execution context",
        "text": "Protect HSC academic performance while preparing for international tests. Plans should account for school hours, local exam cycles, BDT budgets, and access to mentors.",
    },
]

print(json.dumps(student, indent=2, ensure_ascii=False))

{
  "country": "Bangladesh",
  "stage": "HSC / Class 12",
  "target_degree": "Computer Science undergraduate",
  "target_countries": [
    "United States",
    "Canada"
  ],
  "gpa": 3.8,
  "sat": 1320,
  "sat_target": 1500,
  "ielts": 6.5,
  "weekly_hours": 14,
  "budget_bdt": 180000,
  "strengths": [
    "one deployed student portal",
    "school club leadership"
  ],
  "gaps": [
    "testing",
    "research evidence",
    "measured project impact"
  ]
}


## 3. Inspectable BM25-style retrieval

Polaris retrieves evidence deterministically. The model receives only the top matching documents, keeping the reasoning grounded and the retrieval trace easy to inspect.

In [3]:
def tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def retrieve(query, documents, k=3):
    query_terms = tokens(query)
    doc_terms = [tokens(d["title"] + " " + d["text"]) for d in documents]
    n = len(documents)
    document_frequency = Counter(
        term for terms in doc_terms for term in set(terms)
    )
    rows = []
    for document, terms in zip(documents, doc_terms):
        counts = Counter(terms)
        score = 0.0
        for term in query_terms:
            if not counts[term]:
                continue
            inverse_frequency = log((n + 1) / (document_frequency[term] + 0.5))
            score += counts[term] * inverse_frequency
        rows.append((score, document))
    return [document for score, document in sorted(rows, key=lambda x: x[0], reverse=True)[:k]]

query = "SAT moved earlier, protect HSC, prove project impact, limited BDT budget"
evidence = retrieve(query, knowledge_base)
for rank, item in enumerate(evidence, start=1):
    print(f"{rank}. {item['title']} ({item['id']})")
    print("  ", item["text"])

1. Bangladesh execution context (bangladesh)
   Protect HSC academic performance while preparing for international tests. Plans should account for school hours, local exam cycles, BDT budgets, and access to mentors.
2. Project evidence (projects)
   Admissions readers can verify shipped work through public artifacts, repository history, user adoption, independent references, and measured outcomes.
3. Testing evidence (testing)
   Use timed diagnostics, keep an error log by skill, and retest after a focused practice cycle. A score without dated practice evidence is not an actionable plan.


## 4. Structured Gemma 4 helper

The helper requests JSON and validates the returned fields. The web application uses the same compact-contract pattern because shallow schemas are more reliable and easier to audit than unconstrained prose.

In [4]:
def gemma_json(system_instruction, prompt, schema, max_tokens=700):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.2,
            max_output_tokens=max(max_tokens, 1800),
            response_mime_type="application/json",
            response_schema=schema,
        ),
    )
    if response.parsed is not None:
        return response.parsed
    if response.text:
        return json.loads(response.text)
    if response.candidates:
        raise RuntimeError(f"Gemma returned no structured payload ({response.candidates[0].finish_reason})")
    raise RuntimeError("Gemma returned no structured payload")

decision_schema = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "focus": {"type": "string"},
        "next_action": {"type": "string"},
        "evidence": {"type": "string"},
    },
    "required": ["summary", "focus", "next_action", "evidence"],
}

print("Structured helper ready; schema fields:", list(decision_schema["properties"]))

Structured helper ready; schema fields: ['summary', 'focus', 'next_action', 'evidence']


## 5. Decision Twin: stress-test a changed constraint

The scenario is not a request for generic advice. Gemma 4 must compare the new constraint with the complete student context, select the planning focus that should move, and identify the first measurable action.

In [5]:
scenario = "The SAT test date moved six weeks earlier."
evidence_text = "\n".join(
    f"[{i}] {item['title']}: {item['text']}" for i, item in enumerate(evidence, start=1)
)

decision_prompt = f'''
STUDENT
{json.dumps(student, ensure_ascii=False)}

CHANGED CONSTRAINT
{scenario}

RETRIEVED EVIDENCE
{evidence_text}

Return a compact Decision Twin result. Keep each field under 25 words.
The next action must be possible within 24 hours and the evidence must be measurable.
'''

decision = gemma_json(
    "You are Polaris, an academic decision engine for a Bangladeshi student. "
    "Reason about trade-offs across testing, school performance, project evidence, time, and funding. "
    "Never promise admission. Gemma 4 is the only generative model.",
    decision_prompt,
    decision_schema,
)

print(json.dumps(decision, indent=2, ensure_ascii=False))

{
  "summary": "The SAT timeline has compressed, requiring an immediate shift in priority toward intensive, high-frequency testing preparation.",
  "focus": "Accelerated SAT prep and maintaining HSC academic stability while managing the student's limited weekly hours.",
  "next_action": "Complete a full-length, timed, own-brand SAT diagnostic-test within the next 24 hours to establish a baseline.",
  "evidence": "A completed, timed, full-length diagnostic-test score and a detailed error-log by skill-set."
}


## 6. Evidence-to-Action Graph

A student claim is useful only when an admissions reader can verify it. This stage maps a claim through proof, signal, remaining gap, and the next evidence-building action.

In [6]:
evidence_schema = {
    "type": "object",
    "properties": {
        "signal": {"type": "string"},
        "gap": {"type": "string"},
        "next_action": {"type": "string"},
        "verification": {"type": "string"},
    },
    "required": ["signal", "gap", "next_action", "verification"],
}

claim = "I built a student portal used by 120 learners."
proof = "Public repository, deployment analytics, and two teacher references."

evidence_graph = gemma_json(
    "You are the evidence auditor inside Polaris. "
    "Do not mark an unsupported claim as verified. Keep every field under 24 words.",
    f"CLAIM: {claim}\nSUPPLIED PROOF: {proof}\nMap the claim to a verifiable signal, gap, next action, and verification method.",
    evidence_schema,
    500,
)

graph = {
    "claim": claim,
    "proof": proof,
    **evidence_graph,
}
print(json.dumps(graph, indent=2, ensure_ascii=False))

{
  "claim": "I built a student portal used by 120 learners.",
  "proof": "Public repository, deployment analytics, and two teacher references.",
  "signal": "Deployment analytics showing 120 active learners and public repository code for the student portal.",
  "gap": "No direct evidence of user engagement or specific learner identities to confirm the actual usage count.",
  "next_action": "Request access to teacher references or contact them directly to validate the student count and portal usage.",
  "verification": "Cross-reference deployment analytics with teacher testimonials to confirm the 120 learner usage claim."
}


## 7. Bengali reasoning

The public product is fully bilingual. This call demonstrates that the reasoning layer can produce natural Bengali while preserving proper names and admissions acronyms such as SAT, IELTS, GPA, and HSC.

In [7]:
bengali_schema = {
    "type": "object",
    "properties": {
        "diagnosis": {"type": "string"},
        "today": {"type": "string"},
        "metric": {"type": "string"},
    },
    "required": ["diagnosis", "today", "metric"],
}

bengali = gemma_json(
    "আপনি Polaris-এর ভর্তি কৌশলবিদ। সম্পূর্ণ স্বাভাবিক বাংলায় উত্তর দিন। "
    "SAT, IELTS, GPA, HSC এবং বিশ্ববিদ্যালয়ের নাম অপরিবর্তিত রাখুন। "
    "ইংরেজি-বাংলা মিশ্র বাক্য লিখবেন না।",
    f"শিক্ষার্থীর তথ্য: {json.dumps(student, ensure_ascii=False)}\nপরিবর্তন: {scenario}\nএকটি সংক্ষিপ্ত বিশ্লেষণ, আজকের কাজ ও পরিমাপযোগ্য ফলাফল দিন।",
    bengali_schema,
    450,
)

print(json.dumps(bengali, indent=2, ensure_ascii=False))

{
  "diagnosis": "শিক্ষার্থীর বর্তমান SAT score এবং লক্ষ্যমাত্রা হিসেবে নির্ধারিত ১৫০০ স্কোরের মধ্যে ব্যবধান রয়েছে। IELTS 6.5 পাওয়া গেলেও, প্রজেক্টের প্রভাব এবং প্রজেজক্টের ফলাফল পরিমাপ করার ক্ষেত্রে ঘাটতি রয়েছে। SAT প্রস্তুতি প্রস্তুতিতে সময়সীমা সংকুচিত-ই হয়ে গেছে।",
  "today": "SAT প্রস্তুতিতে অতিরিক্ত মনোযোগ দেওয়া এবং প্রজেক্টেরটিရောက်အောင် তোলা এবং প্র পরীক্ষা-পরবর্তী প্রজেক্টের প্রভাব পরিমাপ করার তথ্য সংগ্রহ করা।",
  "metric": "SAT স্কোরের উন্নতি এবং প্রজেক্টের প্রভাব পরিমাপ করার জন্য প্রয়োজনীয় তথ্য সংগ্রহ করা।"
}


## 8. Automated evaluation

These checks do not claim that an automatic metric can replace a counselor. They verify the engineering contract that the UI depends on: required fields, concise actions, measurable evidence, sensitivity to the changed constraint, and Bengali-script coverage.

In [8]:
def has_all(obj, fields):
    return all(isinstance(obj.get(field), str) and obj[field].strip() for field in fields)

decision_fields = ["summary", "focus", "next_action", "evidence"]
evidence_fields = ["signal", "gap", "next_action", "verification"]
bengali_fields = ["diagnosis", "today", "metric"]

evaluation = {
    "decision_schema_valid": has_all(decision, decision_fields),
    "evidence_schema_valid": has_all(evidence_graph, evidence_fields),
    "bengali_schema_valid": has_all(bengali, bengali_fields),
    "decision_mentions_test": bool(re.search(r"SAT|test|diagnostic|score", json.dumps(decision), re.I)),
    "next_action_is_concise": len(decision["next_action"].split()) <= 30,
    "evidence_is_measurable": bool(re.search(r"score|log|completed|users|analytics|date|week|%", decision["evidence"], re.I)),
    "bengali_script_present": bool(re.search(r"[\u0980-\u09FF]", json.dumps(bengali, ensure_ascii=False))),
}
check_total = len(evaluation)
evaluation["passed"] = sum(evaluation.values())
evaluation["total"] = check_total

for name, value in evaluation.items():
    if name not in {"passed", "total"}:
        print(f"{'PASS' if value else 'CHECK'}  {name}")
print(f"\nEngineering checks: {evaluation['passed']}/{evaluation['total']} passed")

PASS  decision_schema_valid
PASS  evidence_schema_valid
PASS  bengali_schema_valid
PASS  decision_mentions_test
PASS  next_action_is_concise
PASS  evidence_is_measurable
PASS  bengali_script_present

Engineering checks: 7/7 passed


## 9. How the notebook maps to the live prototype

| Notebook proof | Live Polaris experience |
|---|---|
| Deterministic retrieval trace | Source-aware Strategist |
| Structured Decision Twin JSON | Interactive before/after roadmap diff |
| Evidence audit | Claim → proof → signal → gap → next action graph |
| English-only diagnostic items | IELTS and SAT Mini Mock Studio |
| Natural-language schedule parsing | Editable weekly Smart Routine |
| Curated official lesson metadata | Embedded IELTS/SAT video learning |
| Bengali structured generation | Full Bengali landing and workspace |

The live application also exposes the complete roadmap, deadlines, university-fit engine, resources, integrations, consultants, community, and family progress views without requiring judge authentication or payment.

## Responsible use and limitations

- Polaris provides planning support, not admission guarantees.
- The displayed probability is a directional planning indicator.
- The IELTS and SAT questions in the application are original, unofficial practice items; results are not official scores.
- Video recommendations come from visible, curated sources rather than an uncontrolled feed.
- Evidence remains labelled incomplete until a human can verify the linked artifact and outcome.
- If the hosted model is unavailable, the application shows a transparent deterministic fallback and never switches to another language model.

## Submission links

- **Live application:** https://polaris-gemma4.vercel.app/
- **Judge workspace:** https://polaris-gemma4.vercel.app/demo
- **Source:** https://github.com/ImtiazHossain-Eshan/polaris-gemma4

Built by **Imtiaz Hossain** and **Mofftasim Hossain Sayem**.